# Safety Guard LoRA - Qwen3 14B

Purpose: train a Nemotron-style content safety classifier LoRA for OpenWebUI content safety and policy filters.

This notebook is intentionally scoped to content safety. It uses Nemotron Safety Guard / Aegis-style safe and unsafe examples and excludes jailbreak-specific samples from the primary safety training mix. Prompt injection gets its own notebook and output contract.

Matching filters:
- `openwebui-safety-filters/content_safety/filter/safety_guard_filter_v3.py`
- `openwebui-safety-filters/policy_violation/filter/safety_filter_company_policy_violation_v1.py`

Expected model output:
```json
{"User Safety": "safe|unsafe", "Response Safety": "safe|unsafe", "Safety Categories": "..."}
```

In [1]:
# Configuration
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_ROOT = PROJECT_ROOT / 'output'
BASE_LLM = 'unsloth/Qwen3-14B-unsloth-bnb-4bit'
MODEL_NAME_BASE = 'safety_guard_qwen3_14b_content_safety'
OUTPUT_BASE_DIR = OUTPUT_ROOT / MODEL_NAME_BASE
CHECKPOINT_DIR = OUTPUT_BASE_DIR / 'train'
LORA_OUTPUT_DIR = OUTPUT_BASE_DIR / 'lora_adapters'

NEMOTRON_DATASET = 'nvidia/Nemotron-Safety-Guard-Dataset-v3'
MAX_SEQ_LENGTH = 4096
SEED = 42

# Keep this LoRA focused on content safety, not prompt injection.
EXCLUDE_NEMOTRON_TAGS = {'jailbreaking'}
MAX_UNSAFE_PER_CATEGORY = 450
SAFE_RATIO_TO_UNSAFE = 0.82

BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 5e-5
TARGET_EPOCHS = 1
WARMUP_STEPS = 50

LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

print(f'Base model: {BASE_LLM}')
print(f'Output: {LORA_OUTPUT_DIR}')

Base model: unsloth/Qwen3-14B-unsloth-bnb-4bit
Output: /workspace/training/safety/output/safety_guard_qwen3_14b_content_safety/lora_adapters


In [2]:
# Imports
import json
import random
import collections
from datasets import load_dataset, concatenate_datasets
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

random.seed(SEED)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# Nemotron / Aegis taxonomy and classifier prompt
SAFETY_TAXONOMY = '''S1: Violence.
S2: Sexual.
S3: Criminal Planning/Confessions.
S4: Guns and Illegal Weapons.
S5: Controlled/Regulated Substances.
S6: Suicide and Self Harm.
S7: Sexual (minor).
S8: Hate/Identity Hate.
S9: PII/Privacy.
S10: Harassment.
S11: Threat.
S12: Profanity.
S13: Needs Caution.
S14: Other.
S15: Manipulation.
S16: Fraud/Deception.
S17: Malware.
S18: High Risk Gov Decision Making.
S19: Political/Misinformation/Conspiracy.
S20: Copyright/Trademark/Plagiarism.
S21: Unauthorized Advice.
S22: Illegal Activity.
S23: Immoral/Unethical.'''

CLASSIFIER_INSTRUCTION = f'''Task: Check if there is unsafe content in the user or agent messages in the below conversation in accordance with our safety policy with the below categories. Provide a 'safe' or 'unsafe' rating for both the user message and the agent response. If there is unsafe content, please also provide a list of violated categories according to our safety policy below.

<BEGIN UNSAFE CONTENT CATEGORIES>
{SAFETY_TAXONOMY}
<END UNSAFE CONTENT CATEGORIES>'''

CATEGORY_MAP = {
    'Violence': 'Violence',
    'Sexual': 'Sexual',
    'Criminal Planning/Confessions': 'Criminal Planning/Confessions',
    'Guns and Illegal Weapons': 'Guns and Illegal Weapons',
    'Controlled/Regulated Substances': 'Controlled/Regulated Substances',
    'Suicide and Self Harm': 'Suicide and Self Harm',
    'Sexual (minor)': 'Sexual (minor)',
    'Hate/Identity Hate': 'Hate/Identity Hate',
    'PII/Privacy': 'PII/Privacy',
    'Harassment': 'Harassment',
    'Threat': 'Threat',
    'Profanity': 'Profanity',
    'Needs Caution': 'Needs Caution',
    'Other': 'Other',
    'Manipulation': 'Manipulation',
    'Fraud/Deception': 'Fraud/Deception',
    'Malware': 'Malware',
    'High Risk Gov Decision Making': 'High Risk Gov Decision Making',
    'Political/Misinformation/Conspiracy': 'Political/Misinformation/Conspiracy',
    'Copyright/Trademark/Plagiarism': 'Copyright/Trademark/Plagiarism',
    'Unauthorized Advice': 'Unauthorized Advice',
    'Illegal Activity': 'Illegal Activity',
    'Immoral/Unethical': 'Immoral/Unethical',
}

In [4]:
# Load and curate Nemotron content-safety data
raw = load_dataset(NEMOTRON_DATASET, split='train')

def keep_content_safety(example):
    if not example.get('prompt') or example.get('prompt') == 'REDACTED':
        return False
    if example.get('language') not in (None, 'en'):
        return False
    if example.get('tag') in EXCLUDE_NEMOTRON_TAGS:
        return False
    return True

dataset = raw.filter(keep_content_safety)
unsafe_pool = dataset.filter(lambda x: x.get('prompt_label') == 'unsafe' or x.get('response_label') == 'unsafe')
safe_pool = dataset.filter(lambda x: x.get('prompt_label') == 'safe' and (not x.get('response_label') or x.get('response_label') == 'safe'))

category_groups = collections.defaultdict(list)
for idx, example in enumerate(unsafe_pool):
    cats = [c.strip() for c in str(example.get('violated_categories', '')).split(',') if c.strip()]
    key = cats[0] if cats else 'Other'
    category_groups[key].append(idx)

selected_unsafe = []
for indices in category_groups.values():
    random.shuffle(indices)
    selected_unsafe.extend(indices[:MAX_UNSAFE_PER_CATEGORY])

nemotron_unsafe = unsafe_pool.select(selected_unsafe)
safe_count = min(len(safe_pool), int(len(nemotron_unsafe) * SAFE_RATIO_TO_UNSAFE))
safe_indices = list(range(len(safe_pool)))
random.shuffle(safe_indices)
nemotron_safe = safe_pool.select(safe_indices[:safe_count])

print(f'Unsafe content-safety examples: {len(nemotron_unsafe)}')
print(f'Safe examples: {len(nemotron_safe)}')
print(f'Excluded tags: {EXCLUDE_NEMOTRON_TAGS}')

Unsafe content-safety examples: 6421
Safe examples: 5265
Excluded tags: {'jailbreaking'}


In [5]:
# Format examples into the OpenWebUI Safety Guard contract
def normalize_categories(value):
    labels = []
    for raw_cat in str(value or '').split(','):
        cat = raw_cat.strip()
        if not cat:
            continue
        labels.append(CATEGORY_MAP.get(cat, cat))
    return ','.join(dict.fromkeys(labels))

def build_classifier_prompt(prompt_text, response_text=None):
    conversation = f'\nuser: {prompt_text}'
    if response_text:
        conversation += f'\nresponse: agent: {response_text}'
    return f'''{CLASSIFIER_INSTRUCTION}

<BEGIN CONVERSATION>
{conversation}
<END CONVERSATION>

Provide your safety assessment for the above conversation in the below output JSON format:
{{"User Safety": safe or unsafe rating for the user message, "Response Safety": safe or unsafe rating for the agent response. Omit if no agent response present. "Safety Categories": a comma-separated list of applicable safety categories from the provided taxonomy. Omit if all safe.}}

Do not include anything other than the output JSON in your response.
Output JSON: '''

def build_classifier_output(prompt_label, response_label=None, categories=''):
    result = {'User Safety': prompt_label or 'safe'}
    if response_label:
        result['Response Safety'] = response_label
    if categories:
        result['Safety Categories'] = categories
    return json.dumps(result, ensure_ascii=False)

def format_example(example):
    user_text = example.get('prompt') or ''
    response_text = example.get('response') or None
    categories = normalize_categories(example.get('violated_categories', ''))
    messages = [
        {'role': 'user', 'content': build_classifier_prompt(user_text, response_text)},
        {'role': 'assistant', 'content': build_classifier_output(example.get('prompt_label'), example.get('response_label'), categories)},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

In [6]:
# Load base model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = concatenate_datasets([nemotron_unsafe, nemotron_safe]).shuffle(seed=SEED)
train_dataset = train_dataset.map(format_example, remove_columns=train_dataset.column_names)
train_dataset = train_dataset.filter(lambda x: 100 < len(x['text']) <= MAX_SEQ_LENGTH * 4)
split = train_dataset.train_test_split(test_size=0.05, seed=SEED)

print(split)

==((====))==  Unsloth 2026.5.7: Fast Qwen3 patching. Transformers: 5.10.0.1.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

unsloth/Qwen3-14B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Map:   0%|          | 0/11686 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11686 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 11100
    })
    test: Dataset({
        features: ['text'],
        num_rows: 585
    })
})


In [7]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


fatal: detected dubious ownership in repository at '/workspace/training/safety'
To add an exception for this directory, call:

	git config --global --add safe.directory /workspace/training/safety


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0
Transformers : 5.10.0.dev0
Torch        : 2.10.0a0+b558c986e8.nv25.11
Triton       : 3.4.0+gitc5d671f9


[transformers] Unsloth 2026.5.7 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [8]:
# Train
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        output_dir=str(CHECKPOINT_DIR),
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type='cosine',
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=100,
        save_steps=200,
        save_total_limit=3,
        bf16=True,
        optim='adamw_8bit',
        seed=SEED,
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/11100 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/multiprocess/popen_fork.py:66: DeprecationWarning: This process (pid=1085) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/585 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/multiprocess/popen_fork.py:66: DeprecationWarning: This process (pid=1085) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,100 | Num Epochs = 1 | Total steps = 694
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 128,450,560 of 14,896,757,760 (0.86% trained)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
100,0.553350,0.411483
200,0.452249,0.384496
300,0.399000,0.373260
400,0.454751,0.367380
500,0.394440,0.363637
600,0.351237,0.361483
694,0.420965,0.361157


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=694, training_loss=0.5173395145523445, metrics={'train_runtime': 9629.6658, 'train_samples_per_second': 1.153, 'train_steps_per_second': 0.072, 'total_flos': 5.523217788624077e+17, 'train_loss': 0.5173395145523445, 'epoch': 1.0})

In [9]:
# Save adapter and training metadata
LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(LORA_OUTPUT_DIR))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR))

metadata = {
    'purpose': 'content_safety',
    'base_model': BASE_LLM,
    'output_contract': 'Nemotron JSON safety classifier',
    'matching_filters': [
        'content_safety/filter/safety_guard_filter_v3.py',
        'policy_violation/filter/safety_filter_company_policy_violation_v1.py',
    ],
    'datasets': [NEMOTRON_DATASET],
    'excluded_tags': sorted(EXCLUDE_NEMOTRON_TAGS),
    'lora': {
        'r': LORA_R,
        'alpha': LORA_ALPHA,
        'target_modules': LORA_TARGET_MODULES,
    },
}
with open(LORA_OUTPUT_DIR / 'safety_guard_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Saved Safety Guard LoRA to {LORA_OUTPUT_DIR}')

[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/safety/output/safety_guard_qwen3_14b_content_safety/lora_adapters/tokenizer_config.json.


Saved Safety Guard LoRA to /workspace/training/safety/output/safety_guard_qwen3_14b_content_safety/lora_adapters
